In [1]:
from pyspark.sql.functions import col, when, lpad
from pyspark.sql.functions import avg
import pyspark.sql.functions as F

StatementMeta(spark1, 4, 2, Finished, Available, Finished, False)

In [2]:
# USPS data

df = spark.read.format("parquet").load("abfss://raw@uspsanalysisfinal.dfs.core.windows.net/uspsanalysis/")
df.count()

StatementMeta(spark1, 4, 3, Finished, Available, Finished, False)

2930142774

In [3]:
# RUCA zipcode data

ruca = spark.read.csv("abfss://raw@uspsanalysisfinal.dfs.core.windows.net/ruca/RUCA-codes-zipcode.csv", header=True,
                        inferSchema=True)
display(ruca.limit(10))

StatementMeta(spark1, 4, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 435b4db7-276d-4cf7-931c-35b652c9fc04)

In [4]:
df.select("orgn_zip_5").show(10)
ruca.select("ZIPCode").show(10)

StatementMeta(spark1, 4, 5, Finished, Available, Finished, False)

+----------+
|orgn_zip_5|
+----------+
|      1741|
|     48340|
|     19720|
|     77581|
|     98405|
|     11747|
|     28206|
|     37601|
|     89503|
|     32114|
+----------+
only showing top 10 rows

+-------+
|ZIPCode|
+-------+
|      1|
|      2|
|      3|
|      4|
|      5|
|      6|
|      7|
|      8|
|      9|
|     10|
+-------+
only showing top 10 rows



In [5]:
# casting ZIP columns as strings
from pyspark.sql.functions import col

df = df.withColumn("orgn_zip_5", col("orgn_zip_5").cast("string")) \
       .withColumn("destn_zip_5", col("destn_zip_5").cast("string"))

ruca = ruca.withColumn("ZIPCode", col("ZIPCode").cast("string"))

StatementMeta(spark1, 4, 6, Finished, Available, Finished, False)

In [6]:
# fixing ZIP formatting
# here we're keeping -1 as 'National' & making sure the zip codes are 5-digit

df = df.withColumn(
    "orgn_zip_5",
    when(col("orgn_zip_5") == "-1", "-1")
    .otherwise(lpad(col("orgn_zip_5"), 5, "0")))

df = df.withColumn(
    "destn_zip_5",
    when(col("destn_zip_5") == "-1", "-1")
    .otherwise(lpad(col("destn_zip_5"), 5, "0")))

ruca = ruca.withColumn("ZIPCode", lpad(col("ZIPCode"), 5, "0"))

StatementMeta(spark1, 4, 7, Finished, Available, Finished, False)

In [7]:
df.select("orgn_zip_5").show(5)
ruca.select("ZIPCode").show(5)

StatementMeta(spark1, 4, 8, Finished, Available, Finished, False)

+----------+
|orgn_zip_5|
+----------+
|     01741|
|     48340|
|     19720|
|     77581|
|     98405|
+----------+
only showing top 5 rows

+-------+
|ZIPCode|
+-------+
|  00001|
|  00002|
|  00003|
|  00004|
|  00005|
+-------+
only showing top 5 rows



**Merging USPS data (zipcodes) to RUCA codes**

In [8]:
df_a = df.alias("df")
ruca_o = ruca.alias("ruca_o")
ruca_d = ruca.alias("ruca_d")

# Origin RUCA table
ruca_o_sel = ruca.select(
    col("ZIPCode").alias("org_zip"),
    col("PrimaryRUCA").cast("int").alias("Origin_RUCA"))

# Destination RUCA table
ruca_d_sel = ruca.select(
    col("ZIPCode").alias("dest_zip"),
    col("PrimaryRUCA").cast("int").alias("Destination_RUCA"))

StatementMeta(spark1, 4, 9, Finished, Available, Finished, False)

In [9]:
# Joining the two datasets so that the USPS data can have urban/rural/suburban classification

merged = df_a.join(ruca_o_sel, df_a.orgn_zip_5 == ruca_o_sel.org_zip,"left").join(
    ruca_d_sel, df_a.destn_zip_5 == ruca_d_sel.dest_zip, "left")

StatementMeta(spark1, 4, 10, Finished, Available, Finished, False)

In [10]:
display(merged.limit(5))

StatementMeta(spark1, 4, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 89c7412c-74c4-4661-8c7f-fae8c53e326d)

In [15]:
#merged = merged.drop("org_zip", "dest_zip")

StatementMeta(spark1, 0, 16, Finished, Available, Finished, False)

In [11]:
# Classification for origin

merged = merged.withColumn(
    "Origin_Area_Type",
    when(col("Origin_RUCA").isin([1,2,3,4,5,6]), "Urban")
    .when(col("Origin_RUCA").isin([7,8,9,10]), "Rural")
    .otherwise("Unknown"))

StatementMeta(spark1, 4, 12, Finished, Available, Finished, False)

In [12]:
# Classification for destination
merged = merged.withColumn(
    "Destination_Area_Type",
    when(col("Destination_RUCA").isin([1,2,3,4,5,6]), "Urban")
    .when(col("Destination_RUCA").isin([7,8,9,10]), "Rural")
    .otherwise("Unknown")
)

StatementMeta(spark1, 4, 13, Finished, Available, Finished, False)

In [13]:
# handling -1 data = national

merged = merged.withColumn(
    "Origin_Area_Type",
    when(col("orgn_zip_5") == "-1", "National")
    .otherwise(col("Origin_Area_Type")))

merged = merged.withColumn(
    "Destination_Area_Type",
    when(col("destn_zip_5") == "-1", "National")
    .otherwise(col("Destination_Area_Type")))

StatementMeta(spark1, 4, 14, Finished, Available, Finished, False)

In [14]:
display(merged.limit(10))

StatementMeta(spark1, 4, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 38c0db8a-e8bb-4829-ac1e-fdc4c877fae5)

In [15]:
# This is zip-level data
zip_level = merged.filter(
    (col("orgn_zip_5") != "-1") &
    (col("destn_zip_5") != "-1"))

# This is national-level data
national_level = merged.filter(
    (col("orgn_zip_5") == "-1") |
    (col("destn_zip_5") == "-1"))

unknown_df = zip_level.filter(
    (col("Origin_Area_Type") == "Unknown") |
    (col("Destination_Area_Type") == "Unknown")
)

# This is the dataframe we'll use for analysis (I didn't include 'Unknown')
analysis_df = zip_level.filter(
    col("Origin_Area_Type").isin(
        "Urban",
        "Rural") &
    col("Destination_Area_Type").isin(
        "Urban",
        "Rural"))

StatementMeta(spark1, 4, 16, Finished, Available, Finished, False)

In [16]:
display(unknown_df.limit(30))

StatementMeta(spark1, 4, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 95263882-eca4-4d9d-bffc-c13cc74bc47f)

In [17]:
#unknown_df.count()

StatementMeta(spark1, 1, 18, Finished, Cancelled, Cancelled, False)

In [20]:
# The relevant columns for analysis
analysis_df_new = analysis_df.select(
    "time_per",
    "orgn_area",
    "orgn_dist",
    "orgn_zip_5",
    "Origin_RUCA",
    "Origin_Area_Type",
    "destn_area",
    "destn_area_name",
    "destn_dist",
    "destn_dist_name",
    "destn_zip_5",
    "Destination_RUCA",
    "Destination_Area_Type",
    "prodt",
    "mo",
    "avg_days_to_delr",
    "score",
    "score_plus_1"
)

StatementMeta(spark1, 4, 21, Finished, Available, Finished, False)

In [21]:
display(analysis_df_new.limit(10))

StatementMeta(spark1, 4, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9445ea75-9336-407a-ae69-24355e22492f)

In [ ]:
# To check for null values

# from pyspark.sql.functions import col, sum, when
# analysis_df_new.select([
#     sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
#     for c in analysis_df_new.columns
# ]).show()

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
#zip_level.count()

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
#analysis_df_new.count()

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
#analysis_df_new.select("time_per").distinct().show()

StatementMeta(, , -1, Waiting, , Waiting, True)

### Analysis

First analysis is for annual.

In [19]:
analysis_df_annual = analysis_df_new.filter(col("time_per") == "Annual")

StatementMeta(spark1, 3, 20, Finished, Available, Finished, False)

Our first step is to analyze urban vs rural service performance.

##### **Destination analysis**

In [37]:
destination_performance_summary_annual = analysis_df_annual.groupby(
    'pstl_yr', 'Destination_Area_Type'
).agg(
    F.count('*').alias('Volume'),
    F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
    (F.avg("score") * 100).alias('Average Score Performance %'),
    (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
    (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
).withColumn("Timeline", F.lit("Annual")
).orderBy('pstl_yr', 'Average Score Performance %')

display(destination_performance_summary_annual)


StatementMeta(spark1, 1, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 834c6895-8fd0-4a86-93b6-8ef42be420fe)

##### **Origin analysis**

In [38]:
origin_performance_summary_annual = analysis_df_annual.groupby('pstl_yr','Origin_Area_Type').agg(
    F.count('*').alias('Volume'),
    F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
    (F.avg("score") * 100).alias('Average Score Performance %'),
    (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
    (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
).withColumn("Timeline", F.lit("Annual")) \
 .orderBy('Average Score Performance %')

display(origin_performance_summary_annual)



StatementMeta(spark1, 1, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 32c0a56c-79d1-4d91-a013-a501e1c083f0)

##### **Origin - Destination analysis**

In [39]:
origin_destination_performance_summary_annual = analysis_df_annual.groupby('pstl_yr', 'Origin_Area_Type', 'Destination_Area_Type').agg(
                                                        F.count('*').alias('Volume'),
                                                        F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
                                                        (F.avg("score") * 100).alias('Average Score Performance %'),
                                                        (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
                                                        (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
                                                            ).withColumn("Timeline", F.lit("Annual")) \
.orderBy('Average Score Performance %')
display(origin_destination_performance_summary_annual)                                                        

StatementMeta(spark1, 1, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d41dbc93-28a3-451c-b72d-e80ac3678e95)

#### **Do different mail product types perform differently in Urban vs Rural destinations?**

In [40]:
product_dest_perf_annual = analysis_df_annual.groupby('pstl_yr', 'prodt', 'Destination_Area_Type').agg(
                                                        F.count('*').alias('Volume'),
                                                        F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
                                                        (F.avg("score") * 100).alias('Average Score Performance %'),
                                                        (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
                                                        (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
                                                            ).withColumn("Timeline", F.lit("Annual")) \
.orderBy('Average Score Performance %')
display(product_dest_perf_annual)                                                        

StatementMeta(spark1, 1, 41, Submitted, Running, Running, False)

# Monthly

In [20]:
analysis_df_monthly = analysis_df_new.filter(col("time_per") == "Monthly")

StatementMeta(spark1, 3, 21, Finished, Available, Finished, False)

Destination Analysis 

In [ ]:
destination_performance_summary_monthly = analysis_df_monthly.groupby('pstl_yr', 'mo','Destination_Area_Type').agg(
    F.count('*').alias('Volume'),
    F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
    (F.avg("score") * 100).alias('Average Score Performance %'),
    (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
    (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
).withColumn("Timeline", F.lit("Monthly")) \
 .orderBy('Average Score Performance %')

display(destination_performance_summary_monthly)

StatementMeta(, , -1, Waiting, , Waiting, True)

Origin Analysis

In [ ]:
origin_performance_summary_monthly = analysis_df_monthly.groupby('pstl_yr', 'mo','Origin_Area_Type').agg(
    F.count('*').alias('Volume'),
    F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
    (F.avg("score") * 100).alias('Average Score Performance %'),
    (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
    (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
).withColumn("Timeline", F.lit("Monthly")) \
 .orderBy('Average Score Performance %')

display(origin_performance_summary_monthly)

StatementMeta(, , -1, Waiting, , Waiting, True)

Origin-Destination Analysis

In [ ]:
origin_destination_performance_summary_monthly = analysis_df_monthly.groupby('pstl_yr', 'mo','Origin_Area_Type', 'Destination_Area_Type').agg(
                                                        F.count('*').alias('Volume'),
                                                        F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
                                                        (F.avg("score") * 100).alias('Average Score Performance %'),
                                                        (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
                                                        (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
                                                            ).withColumn("Timeline", F.lit("Monthly")) \
.orderBy('Average Score Performance %')
display(origin_destination_performance_summary_monthly)  

StatementMeta(, , -1, Waiting, , Waiting, True)

Do different mail product types perform differently in Urban vs Rural destinations?

In [ ]:
product_dest_perf_monthly = analysis_df_monthly.groupby('pstl_yr', 'mo','prodt', 'Destination_Area_Type').agg(
                                                        F.count('*').alias('Volume'),
                                                        F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
                                                        (F.avg("score") * 100).alias('Average Score Performance %'),
                                                        (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
                                                        (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
                                                            ).withColumn("Timeline", F.lit("Monthly")) \
.orderBy('Average Score Performance %')
display(product_dest_perf_monthly)   

StatementMeta(, , -1, Waiting, , Waiting, True)

# Quarterly

In [21]:
analysis_df_annual = analysis_df_new.filter(col("time_per") == "Quarterly")

StatementMeta(spark1, 3, 22, Finished, Available, Finished, False)

Destination Analysis

In [ ]:
destination_performance_summary_quarterly = analysis_df_quarterly.groupby('pstl_yr', 'pstl_qtr','Destination_Area_Type').agg(
    F.count('*').alias('Volume'),
    F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
    (F.avg("score") * 100).alias('Average Score Performance %'),
    (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
    (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
).withColumn("Timeline", F.lit("Quarterly")) \
 .orderBy('Average Score Performance %')

display(destination_performance_summary_quarterly)

Origin Analysis

In [ ]:
origin_performance_summary_quarterly = analysis_df_quarterly.groupby('pstl_yr', 'pstl_qtr','Origin_Area_Type').agg(
    F.count('*').alias('Volume'),
    F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
    (F.avg("score") * 100).alias('Average Score Performance %'),
    (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
    (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
).withColumn("Timeline", F.lit("Quarterly")) \
 .orderBy('Average Score Performance %')

display(origin_performance_summary_quarterly)

Origin-Destination Analysis

In [ ]:
origin_destination_performance_summary_quarterly = analysis_df_quarterly.groupby('pstl_yr', 'pstl_qtr','Origin_Area_Type', 'Destination_Area_Type').agg(
                                                        F.count('*').alias('Volume'),
                                                        F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
                                                        (F.avg("score") * 100).alias('Average Score Performance %'),
                                                        (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
                                                        (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
                                                            ).withColumn("Timeline", F.lit("Quarterly")) \
.orderBy('Average Score Performance %')
display(origin_destination_performance_summary_quarterly)  

Do different mail product types perform differently in Urban vs Rural destinations?

In [ ]:
product_dest_perf_quarterly = analysis_df_quarterly.groupby('pstl_yr', 'pstl_qtr','prodt', 'Destination_Area_Type').agg(
                                                        F.count('*').alias('Volume'),
                                                        F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
                                                        (F.avg("score") * 100).alias('Average Score Performance %'),
                                                        (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
                                                        (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
                                                            ).withColumn("Timeline", F.lit("Quarterly")) \
.orderBy('Average Score Performance %')
display(product_dest_perf_quarterly)   

# Weekly

In [22]:
analysis_df_annual = analysis_df_new.filter(col("time_per") == "Weekly")

StatementMeta(spark1, 3, 23, Finished, Available, Finished, False)

Destination Analysis

In [ ]:
destination_performance_summary_weekly = analysis_df_weekly.groupby('Destination_Area_Type').agg(
    F.count('*').alias('Volume'),
    F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
    (F.avg("score") * 100).alias('Average Score Performance %'),
    (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
    (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
).withColumn("Timeline", F.lit("Weekly")) \
 .orderBy('Average Score Performance %')

display(destination_performance_summary_weekly)

Origin Analysis

In [ ]:
origin_performance_summary_weekly = analysis_df_weekly.groupby('Origin_Area_Type').agg(
    F.count('*').alias('Volume'),
    F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
    (F.avg("score") * 100).alias('Average Score Performance %'),
    (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
    (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
).withColumn("Timeline", F.lit("Weekly")) \
 .orderBy('Average Score Performance %')

display(origin_performance_summary_weekly)

Origin-Destination Analysis

In [ ]:
origin_destination_performance_summary_weekly = analysis_df_weekly.groupby('Origin_Area_Type', 'Destination_Area_Type').agg(
                                                        F.count('*').alias('Volume'),
                                                        F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
                                                        (F.avg("score") * 100).alias('Average Score Performance %'),
                                                        (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
                                                        (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
                                                            ).withColumn("Timeline", F.lit("Weekly")) \
.orderBy('Average Score Performance %')
display(origin_destination_performance_summary_weekly)  

Do different mail product types perform differently in Urban vs Rural destinations?

In [ ]:
product_dest_perf_weekly= analysis_df_weekly.groupby('prodt', 'Destination_Area_Type').agg(
                                                        F.count('*').alias('Volume'),
                                                        F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
                                                        (F.avg("score") * 100).alias('Average Score Performance %'),
                                                        (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
                                                        (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
                                                            ).withColumn("Timeline", F.lit("Weekly")) \
.orderBy('Average Score Performance %')
display(product_dest_perf_weekly)   

In [25]:
zip_performance = analysis_df_new.groupby(
    'destn_zip_5', 'Destination_Area_Type'
).agg(
    F.count('*').alias('Volume'),
    F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
    (F.avg("score") * 100).alias('Average Score Performance %')
).orderBy('Average Score Performance %')
display(zip_performance)

StatementMeta(spark1, 3, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 01445113-90dc-49ff-823c-2bcce15f561b)

In [26]:
district_performance = analysis_df_new.groupby(
    'destn_dist', 'destn_area', 'Destination_Area_Type'
).agg(
    F.count('*').alias('Volume'),
    F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
    (F.avg("score") * 100).alias('Average Score Performance %')
).orderBy('Average Score Performance %')

display(district_performance)

StatementMeta(spark1, 3, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8deb94ea-3beb-439c-abee-b03d2a26f232)

In [27]:
product_dest_perf_annual = analysis_df_annual.withColumn('mail_class',
    F.when(F.col('prodt').contains('First-Class'), 'First-Class')
     .when(F.col('prodt').contains('Marketing'), 'Marketing Mail')
     .when(F.col('prodt').contains('Periodicals'), 'Periodicals')
     .otherwise('Other')
).groupby('pstl_yr', 'mail_class', 'prodt', 'Destination_Area_Type').agg(
    F.count('*').alias('Volume'),
    F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
    (F.avg("score") * 100).alias('Average Score Performance %'),
    (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %'),
    (F.percentile_approx("score", 0.5) * 100).alias('Median Score %')
).withColumn("Timeline", F.lit("Annual")
).orderBy('pstl_yr', 'mail_class', 'Average Score Performance %')

display(product_dest_perf_annual)

StatementMeta(spark1, 3, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 191cc030-643e-4028-bd57-abfbc489020a)

In [22]:
district_performance = analysis_df_new.groupby(
    'destn_dist', 'destn_dist_name', 'destn_area', 'destn_area_name', 'Destination_Area_Type'
).agg(
    F.count('*').alias('Volume'),
    F.avg("avg_days_to_delr").alias('Average Days to Deliver'),
    (F.avg("score") * 100).alias('Average Score Performance %'),
    (F.avg("score_plus_1") * 100).alias('Average Score Plus 1 %')
).orderBy('Average Score Performance %')

display(district_performance)

StatementMeta(spark1, 4, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 21fe8a51-aa12-4970-817e-3d1a9cb1cf6e)